<div dir="rtl" align="right">

# NB22C — تشخیص و آزمایش بهبود بازیابی

فایل را با نام `22c_retrieval_diagnosis_and_improvement.ipynb` در `notebooks/phase2/` قرار بده و **Restart Kernel → Run All** اجرا کن. هیچ سل NB22B یا کرنل قبلی لازم نیست. Root از مسیر مخزن پیدا می‌شود؛ در صورت نیاز `RAG_FINANCE_ROOT` را تعیین کن.

**۶ اقدام، هرکدام سه زیرکار، در شش بستهٔ وابسته؛ به‌اضافهٔ Imports، Bootstrap و مکانیک مشترک.** شرط پایان: گزارش قبل/بعد، گزینهٔ حفظ‌شده، سقف پوشش و گام بعدی؛ بهبود یا پذیرش تضمین نیست.

شاهد خوانده‌شده: گزارش واقعی A05 و A06؛ Top100 کامل بوده و روی Development، Recall@10 از ۰٫۶۶۷۶۷ به ۰٫۶۶۵۵۹ و nDCG@10 از ۰٫۵۹۸۸۳ به ۰٫۵۹۷۲۶ رسیده؛ انتخاب‌نشدن آن ناشی از نتیجهٔ ضعیف‌تر بوده است. منتخب قبلی `cached_rrf_k10_w0.75` است.

آزمایش تازه: ۱۸ ترکیب ثابت از رتبه‌های Hybrid100 و Local100 موجود، همراه محدودیت حداکثر یک یا دو ورودی از خارج Pool50 در Top10. ورودی مدل، رأی و Gold تغییر نمی‌کند. **API، دانلود و اجرای دوبارهٔ مدل: صفر.** خروجی حجیم A05 مصرف می‌شود؛ فایل Cache خام مدل لازم نیست.

انتخاب روی بخش tuning از Development و یک بررسی داخلی روی بخش validation با مؤلفه‌های منبع جدا انجام می‌شود. هر دو قبلاً در Development دیده شده‌اند؛ این validation آزمون مستقل نهایی نیست. Test تاریخی برای آزمایش تازه ارزیابی نمی‌شود؛ ردیف‌های آن تنها در کنترل هش/جمعیت و نگاشت Gold خوانده می‌شوند.

اگر فایل پین‌شده موجود نباشد، مسیر دقیق چاپ و مراحل وابسته مسدود می‌شوند؛ هیچ بازسازی یا ارسال تازه‌ای انجام نمی‌شود. مسیر جدید خروجی `reports/phase2/quality_improvement/NB22C.v1/` است؛ گزارش‌های قبلی حفظ می‌شوند. تکرار اجرای این نوت‌بوک محاسبات سبک رتبه و ارزیابی را تکرار می‌کند، فایل یکسان را مصرف می‌کند و هیچ مدل/API را اجرا نمی‌کند.

خروجی نهایی هر شش اقدام را بخوان؛ پایان بدون خطای Python با تکمیل اقدام یا پذیرش کیفیت یکی نیست. گزارش‌های BLOCKED/FAILED نیز چاپ می‌شوند. تعویق اقدام۴۸، مسائل باز، عدم‌پذیرش Release و حدود NB23 محفوظ‌اند.

</div>

<div dir="rtl" align="right">

### NB22C-C001 — Imports

فقط Imports استاندارد؛ هیچ عملیات فایل یا شبکه ندارد.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [1]:
# NB22C-C001 — Imports only; standard library, no file/network/model operations.
import collections
import hashlib
import json
import math
import os
import platform
import re
import statistics
import tempfile
import time
from pathlib import Path


<div dir="rtl" align="right">

### NB22C-C002 — Bootstrap

فقط دریافت مسیر و تنظیمات صریح و هویت فایل؛ بدون نوشتن یا اجرا.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [2]:
# NB22C-C002 — Read-only bootstrap and explicit configuration.
PINS = {'C056': {'path': 'reports/phase2/release/NB22.C056.retrieval_verification/C056.independent_retrieval_metric_verification.ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22.json', 'file_bytes_sha256': 'ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22', 'bytes': 1180779, 'write_status': 'CREATED', 'reload_validation': 'PASS'}, 'A05': {'path': 'reports/phase2/quality_improvement/NB22B.v1/A05.result.e97bb86424371499c5c73f6c90894aca3d7704cddc0de0463541e6cb5f941833.json', 'file_bytes_sha256': 'e97bb86424371499c5c73f6c90894aca3d7704cddc0de0463541e6cb5f941833'}, 'A06': {'path': 'reports/phase2/quality_improvement/NB22B.v1/A06.result.7c4539e2a1957ec83b66abf2270c122e3c484eab324d7a8badd32baf60fdd2bd.json', 'file_bytes_sha256': '7c4539e2a1957ec83b66abf2270c122e3c484eab324d7a8badd32baf60fdd2bd'}}

CONFIG = {
    'version': 'NB22C_CACHED_DEPTH100_FUSION_V1',
    'seed': 'NB22C_20261030_SOURCE_COMPONENT_SPLIT_V1',
    'validation_percent': 20,
    'weights': [0.50, 0.75, 0.90],
    'constants': [10, 30],
    'newcomer_limits': [1, 2],
    'minimum_components_per_partition': 30,
    'thresholds': {'recall_at_10': 0.85, 'ndcg_at_10': 0.75},
    'api_calls': 0, 'model_inference': False, 'downloads': False,
    'historical_test_evaluation': False,
}
_start = Path(os.environ.get('RAG_FINANCE_ROOT', str(Path.cwd()))).expanduser().resolve()
ROOT = next((p for p in [_start, *_start.parents]
    if (p / 'notebooks/phase2').is_dir() and
    (p / 'data/Road_Maps/RAG_finance_phase2_modular_notebooks_execution_plan.md').is_file()), None)
STATE = {'actions': {}, 'artifacts': [], 'memory': {}, 'errors': [], 'inputs': {},
         'api_calls': 0, 'model_pairs': 0, 'network_calls': 0}
print(json.dumps({'stage': 'BOOTSTRAP', 'root_resolved': ROOT is not None,
    'experiment': CONFIG['version'], 'new_api_calls': 0, 'new_model_pairs': 0,
    'historical_test_evaluation': False}, ensure_ascii=False, indent=2))

# BEGIN SELF IDENTITY — bind saved notebook code, independent of kernel history.
NOTEBOOK_CODE_HASH = None
SAVED_CODE_VALID = False
if ROOT is not None:
    _self = ROOT / 'notebooks/phase2/22c_retrieval_diagnosis_and_improvement.ipynb'
    if _self.is_file():
        _nb = json.loads(_self.read_bytes())
        _codes = [''.join(c['source']) if isinstance(c['source'], list) else c['source']
                  for c in _nb['cells'] if c['cell_type'] == 'code']
        NOTEBOOK_CODE_HASH = hashlib.sha256(json.dumps(_codes, ensure_ascii=False,
            sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()
        SAVED_CODE_VALID = NOTEBOOK_CODE_HASH == _nb.get('metadata', {}).get('nb22c', {}).get('code_source_sha256')


{
  "stage": "BOOTSTRAP",
  "root_resolved": true,
  "experiment": "NB22C_CACHED_DEPTH100_FUSION_V1",
  "new_api_calls": 0,
  "new_model_pairs": 0,
  "historical_test_evaluation": false
}


<div dir="rtl" align="right">

### NB22C-C003 — مکانیک مشترک

تعریف کنترل هش، نگاشت، ذخیرهٔ Atomic و رتبه‌سازی؛ بدون اجرای مدل.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [3]:
# NB22C-C003 — Shared mechanics; no historical notebook is executed.
class InputGap(RuntimeError):
    pass

def require(ok, reason):
    if not ok:
        raise InputGap(reason)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')

def digest(raw):
    return hashlib.sha256(raw).hexdigest()

def decode(raw):
    def pairs(items):
        out = {}
        for k, v in items:
            if k in out:
                raise InputGap('DUPLICATE_JSON_KEY:' + k)
            out[k] = v
        return out
    def reject(x):
        raise InputGap('NONFINITE_JSON:' + x)
    return json.loads(raw, object_pairs_hook=pairs, parse_constant=reject)

def path_for(relative):
    require(ROOT is not None, 'PROJECT_ROOT_REQUIRED:run_inside_repository_or_set_RAG_FINANCE_ROOT')
    p = Path(relative)
    require(not p.is_absolute() and '..' not in p.parts, 'PROJECT_RELATIVE_PATH_REQUIRED')
    target = (ROOT / p).resolve()
    require(target.is_relative_to(ROOT), 'PATH_OUTSIDE_PROJECT')
    return target

def binding_hash(binding):
    values = {binding[k] for k in ('file_bytes_sha256', 'sha256') if k in binding}
    require(len(values) == 1, 'UNAMBIGUOUS_BYTE_HASH_REQUIRED')
    h = next(iter(values))
    require(isinstance(h, str) and re.fullmatch('[a-f0-9]{64}', h), 'INVALID_HASH')
    return h

def checked_bytes(binding):
    p = path_for(binding['path'])
    require(p.is_file(), 'REQUIRED_FILE_NOT_AVAILABLE:' + binding['path'])
    before = p.stat()
    raw = p.read_bytes()
    after = p.stat()
    require((before.st_size, before.st_mtime_ns, before.st_ino) ==
            (after.st_size, after.st_mtime_ns, after.st_ino), 'SOURCE_CHANGED_DURING_READ')
    require(digest(raw) == binding_hash(binding), 'INPUT_BYTE_HASH_MISMATCH:' + binding['path'])
    STATE['inputs'][binding['path']] = dict(binding)
    return raw

def read_json(binding):
    return decode(checked_bytes(binding))

def read_rows(binding):
    raw = checked_bytes(binding)
    require(all(line.strip() for line in raw.splitlines()), 'BLANK_JSONL_ROW')
    return [decode(line) for line in raw.splitlines()]

def keyed(rows, field):
    out = {}
    for row in rows:
        require(isinstance(row, dict) and isinstance(row.get(field), str) and
                row[field] not in out, 'MISSING_OR_DUPLICATE_ID:' + field)
        out[row[field]] = row
    return out

def save(stem, value, rows=False):
    folder = path_for('reports/phase2/quality_improvement/NB22C.v1')
    folder.mkdir(parents=True, exist_ok=True)
    raw = b''.join(canonical(r) + b'\n' for r in value) if rows else canonical(value)
    suffix = 'jsonl' if rows else 'json'
    p = folder / (stem + '.' + digest(raw) + '.' + suffix)
    if not p.exists():
        fd, name = tempfile.mkstemp(dir=folder, prefix='.nb22c-')
        try:
            with os.fdopen(fd, 'wb') as f:
                f.write(raw); f.flush(); os.fsync(f.fileno())
            try:
                os.link(name, p)
            except FileExistsError:
                pass
        finally:
            Path(name).unlink(missing_ok=True)
    actual = p.read_bytes()
    require(actual == raw, 'OUTPUT_COLLISION_OR_RELOAD_FAILURE')
    if rows:
        require([decode(x) for x in actual.splitlines()] == value, 'OUTPUT_ROW_RELOAD_FAILURE')
    else:
        require(decode(actual) == value, 'OUTPUT_RELOAD_FAILURE')
    receipt = {'path': p.relative_to(ROOT).as_posix(), 'file_bytes_sha256': digest(raw),
               'bytes': len(raw), 'reload_validation': 'PASS'}
    STATE['artifacts'].append(receipt)
    return receipt

def metrics(order, gold):
    require(gold and len(order) == len(set(order)), 'INVALID_METRIC_POPULATION')
    flags = [int(x in gold) for x in order[:50]]
    out = {}
    for k in (5, 10):
        hits = sum(flags[:k])
        idcg = math.fsum(1 / math.log2(i + 2) for i in range(min(k, len(gold))))
        out.update({f'hit_at_{k}': float(hits > 0), f'precision_at_{k}': hits / k,
                    f'recall_at_{k}': hits / len(gold),
                    f'ndcg_at_{k}': math.fsum(v / math.log2(i + 2) for i, v in enumerate(flags[:k])) / idcg})
    first = next((i + 1 for i, v in enumerate(flags) if v), None)
    out['reciprocal_rank'] = 1 / first if first else 0.0
    return out

def mean_metric(values):
    rows = list(values)
    require(bool(rows), 'EMPTY_METRIC_COHORT')
    return {k: math.fsum(r[k] for r in rows) / len(rows) for k in sorted(rows[0])}

def rank_ids(row, count):
    c = row['candidates']
    require(len(c) == count and [r['rank'] for r in c] == list(range(1, count + 1)), 'RANK_SEQUENCE_OR_DEPTH_CONFLICT')
    ids = [r['chunk_id'] for r in c]
    require(len(ids) == len(set(ids)), 'DUPLICATE_RANKED_CHUNK')
    return ids

def run_action(number, fn, dependencies=()):
    blocked = [d for d in dependencies if STATE['actions'].get(d, {}).get('status') != 'COMPLETE_IN_SCOPE']
    if blocked:
        out = {'status': 'BLOCKED_BY_DEPENDENCY', 'dependencies': blocked}
    else:
        try:
            out = fn()
            out['status'] = 'COMPLETE_IN_SCOPE'
        except InputGap as error:
            out = {'status': 'BLOCKED_INPUT_OR_CONTRACT', 'reason': str(error)}
        except Exception as error:
            # Real implementation errors remain distinguishable from missing input.
            out = {'status': 'FAILED_IMPLEMENTATION', 'error_type': type(error).__name__}
            STATE['errors'].append({'action': number, 'error_type': type(error).__name__})
    STATE['actions'][number] = out
    print(json.dumps({'action': number, **out, 'new_api_calls': 0}, ensure_ascii=False, indent=2))
    return out

def components(dev):
    parent = {q: q for q in dev['ids']}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    seen = {}
    for q in dev['ids']:
        for cid in dev['gold'][q]:
            sid = dev['chunks'][cid].get('source_id')
            require(isinstance(sid, str) and sid.strip(), 'GOLD_SOURCE_PROVENANCE_REQUIRED')
            if sid in seen:
                a, b = find(q), find(seen[sid])
                parent[max(a, b)] = min(a, b)
            else:
                seen[sid] = q
    return {q: find(q) for q in dev['ids']}

def partition(component_map):
    labels = {c: ('validation' if int(digest((CONFIG['seed'] + c).encode())[:16], 16) % 100
                    < CONFIG['validation_percent'] else 'tuning') for c in set(component_map.values())}
    out = {name: sorted(q for q, c in component_map.items() if labels[c] == name)
           for name in ('tuning', 'validation')}
    for name, ids in out.items():
        require(len({component_map[q] for q in ids}) >= CONFIG['minimum_components_per_partition'],
                'INSUFFICIENT_SOURCE_COMPONENTS:' + name)
    return out

def eligible(candidate, reference):
    return all(candidate[k] >= reference[k] - 1e-12 for k in ('recall_at_10', 'hit_at_10')) and candidate['ndcg_at_10'] > reference['ndcg_at_10'] + 1e-12


def derived_order(hybrid, local, incumbent, spec):
    require(len(hybrid) == len(local) == 100 and set(hybrid) == set(local), 'DEPTH100_POOL_CONFLICT')
    require(len(incumbent) == 50 and set(incumbent).issubset(hybrid), 'INCUMBENT_POOL_CONFLICT')
    hr = {c: i + 1 for i, c in enumerate(hybrid)}
    lr = {c: i + 1 for i, c in enumerate(local)}
    k, w = spec['k'], spec['weight']
    ranked = sorted(hybrid, key=lambda c: (-(w / (k + lr[c]) + (1 - w) / (k + hr[c])), c))
    if spec['family'] == 'quota100':
        old_pool = set(incumbent)
        top = []; admitted = 0
        for c in ranked:
            if c not in old_pool and admitted >= spec['newcomer_limit']:
                continue
            top.append(c)
            admitted += int(c not in old_pool)
            if len(top) == 10:
                break
        chosen = set(top)
        ranked = [c for c in ranked if c in chosen] + [c for c in ranked if c not in chosen]
    require(len(ranked) == 100 and len(set(ranked)) == 100, 'DERIVED_RANK_POPULATION_CONFLICT')
    return ranked


<div dir="rtl" align="right">

### NB22C-C004 — اقدام ۱ — اتصال ورودی‌ها

۱٫۱ گزارش‌های پین‌شده؛ ۱٫۲ کنترل Gold و جمعیت؛ ۱٫۳ مصرف Local100 و منتخب50. شرط پایان: تطبیق هش، شناسه، معیار و جمعیت کامل.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [4]:
# NB22C-C004 — Action 1: bind exact prior results and Development inputs.
def action01():
    require(SAVED_CODE_VALID, 'SAVED_NOTEBOOK_CODE_IDENTITY_REQUIRED')
    previous = {name: read_json(b) for name, b in PINS.items()}
    a05, a06, verified = previous['A05'], previous['A06'], previous['C056']
    require(a05['action'] == 5 and a06['action'] == 6, 'PRIOR_ACTION_SCHEMA_CONFLICT')
    require(verified['independent_recomputation_complete'] is True and
            verified['all_recorded_metric_comparisons_match'] is True, 'C056_VERIFICATION_REQUIRED')
    require(a06['selected']['selected_variant'] == 'cached_rrf_k10_w0.75', 'PREVIOUS_SELECTION_CONFLICT')
    bindings = verified['input_file_receipts']
    missing = [b['path'] for b in [*bindings.values(), a05['results']]
               if not path_for(b['path']).is_file()]
    require(not missing, 'EXACT_INPUT_FILES_REQUIRED:' + json.dumps(missing))
    chunks = keyed(read_rows(bindings['chunks']), 'chunk_id')
    golden = keyed(read_rows(bindings['golden']), 'candidate_id')
    hybrid_all = keyed(read_rows(bindings['hybrid']), 'candidate_id')
    reranker_all = keyed(read_rows(bindings['reranker']), 'candidate_id')
    require(len(chunks) == 3488 and len(golden) == 3813, 'FROZEN_POPULATION_CONFLICT')
    require(set(golden) == set(hybrid_all) == set(reranker_all), 'QUERY_ID_POPULATION_CONFLICT')
    require(collections.Counter(r['split'] for r in hybrid_all.values()) ==
            {'development': 2664, 'test': 1149}, 'FROZEN_SPLIT_CONFLICT')
    locators = collections.defaultdict(set); types = collections.defaultdict(set)
    for cid, row in chunks.items():
        for m in row['member_locators']:
            key = canonical(m['locator'])
            locators[key].add(cid); types[key].add(row['source_type'])
    gold = {}
    for q, row in golden.items():
        relevant = set()
        for e in row['evidence_locators']:
            key = canonical(e['source_locator'])
            require(bool(locators[key]) and bool(types[key] & set(row['source_types'])), 'GOLD_LOCATOR_OR_TYPE_CONFLICT:' + q)
            relevant.update(locators[key])
        require(bool(relevant), 'EMPTY_GOLD:' + q)
        gold[q] = relevant
    logical = hashlib.sha256()
    for q in sorted(gold):
        raw = canonical({'candidate_id': q, 'relevant_chunk_ids': sorted(gold[q])})
        logical.update(len(raw).to_bytes(8, 'big')); logical.update(raw)
    require(logical.hexdigest() == verified['gold_mapping']['recomputed_sha256'], 'LOGICAL_GOLD_IDENTITY_CONFLICT')
    ids = sorted(q for q in golden if hybrid_all[q]['split'] == 'development')
    dev = {'ids': ids, 'chunks': chunks, 'gold': {q: gold[q] for q in ids},
           'source_types': {q: sorted(golden[q]['source_types']) for q in ids},
           'hybrid': {}, 'baseline': {}, 'local': {}, 'incumbent': {}}
    for q in ids:
        h, r = hybrid_all[q], reranker_all[q]
        require(r['split'] == 'development', 'DEVELOPMENT_SPLIT_CONFLICT')
        ho, ro = rank_ids(h, 100), rank_ids(r, 50)
        require(set(ro) == set(ho[:50]) and set(ho).issubset(chunks), 'SAME50_POOL_OR_UNKNOWN_CHUNK_CONFLICT')
        scores = [float(c['reranker_score']) for c in r['candidates']]
        require(all(math.isfinite(x) for x in scores) and all(a >= b for a, b in zip(scores, scores[1:])), 'INVALID_PRIOR_SCORES')
        dev['hybrid'][q] = ho; dev['baseline'][q] = ro
    # Historical Test rows were read only to verify frozen population and logical mapping.
    # No Test metrics, selection, retrieval inference or answer generation occurs here.
    del golden, hybrid_all, reranker_all, gold
    prior_variants = {'local_onnx_depth100': 'local', 'cached_rrf_k10_w0.75': 'incumbent'}
    # Stream the 140 MB prior artifact, validate its complete byte hash and retain only two orders.
    b = a05['results']; p = path_for(b['path']); before = p.stat(); h = hashlib.sha256()
    seen = {v: set() for v in prior_variants}
    with p.open('rb') as f:
        for raw in f:
            h.update(raw); row = decode(raw)
            if row.get('variant') not in prior_variants:
                continue
            name, q = row['variant'], row['candidate_id']
            require(q in dev['gold'] and q not in seen[name] and row['split'] == 'development', 'CACHED_VARIANT_COHORT_CONFLICT')
            seen[name].add(q); order = row['ranked_chunk_ids']
            expected = dev['hybrid'][q] if name == 'local_onnx_depth100' else dev['hybrid'][q][:50]
            require(len(order) == len(expected) and set(order) == set(expected), 'CACHED_VARIANT_POOL_CONFLICT')
            require(all(abs(v - row['metrics'][k]) <= 1e-12 for k, v in metrics(order, dev['gold'][q]).items()), 'CACHED_PER_QUERY_METRIC_CONFLICT')
            dev[prior_variants[name]][q] = order
    after = p.stat()
    require((before.st_size, before.st_mtime_ns, before.st_ino) ==
            (after.st_size, after.st_mtime_ns, after.st_ino), 'CACHED_VARIANTS_CHANGED_DURING_READ')
    require(h.hexdigest() == binding_hash(b), 'CACHED_VARIANTS_BYTE_HASH_CONFLICT')
    STATE['inputs'][b['path']] = b
    require(all(s == set(ids) for s in seen.values()), 'COMPLETE_LOCAL100_AND_INCUMBENT_CACHE_REQUIRED_NO_INFERENCE_FALLBACK')
    for field, expected in [('baseline', a06['selected']['baseline_metrics']),
                            ('incumbent', a06['selected']['selected_development_metrics']),
                            ('local', next(x['metrics'] for x in a06['comparisons'] if x['variant'] == 'local_onnx_depth100'))]:
        actual = mean_metric(metrics(dev[field][q], dev['gold'][q]) for q in ids)
        require(all(abs(actual[k] - expected[k]) <= 1e-12 for k in actual), 'PRIOR_AGGREGATE_REPLAY_CONFLICT:' + field)
    STATE['memory'].update(dev=dev, previous=previous)
    return {'subtasks': {'1.1': 'BOUND_PRIOR_REPORTS', '1.2': 'VERIFIED_GOLD_AND_DEVELOPMENT',
                         '1.3': 'REUSED_COMPLETE_LOCAL100_AND_SELECTED50'},
            'development_queries': len(ids), 'local_model_pairs_recomputed': 0,
            'historical_artifacts_modified': False,
            'previous_top100_development': next(x for x in a06['comparisons'] if x['variant'] == 'local_onnx_depth100')}
A01 = run_action(1, action01)


{
  "action": 1,
  "subtasks": {
    "1.1": "BOUND_PRIOR_REPORTS",
    "1.2": "VERIFIED_GOLD_AND_DEVELOPMENT",
    "1.3": "REUSED_COMPLETE_LOCAL100_AND_SELECTED50"
  },
  "development_queries": 2664,
  "local_model_pairs_recomputed": 0,
  "historical_artifacts_modified": false,
  "previous_top100_development": {
    "delta": {
      "hit_at_10": 0.0033783783783782884,
      "hit_at_5": 0.0030030030030030463,
      "ndcg_at_10": -0.0015753327535392714,
      "ndcg_at_5": -0.0010780522709706863,
      "precision_at_10": -0.00048798798798799226,
      "precision_at_5": -0.0007507507507507338,
      "recall_at_10": -0.0020851208351209127,
      "recall_at_5": -0.001070266695266664,
      "reciprocal_rank": 0.0014876814237273495
    },
    "eligible_by_frozen_point_selection_rule": false,
    "excluded_queries": 0,
    "metrics": {
      "hit_at_10": 0.8532282282282282,
      "hit_at_5": 0.7946696696696697,
      "ndcg_at_10": 0.5972588973181474,
      "ndcg_at_5": 0.572198463005388,
      

<div dir="rtl" align="right">

### NB22C-C005 — اقدام ۲ — تشخیص و تفکیک داخلی

۲٫۱ سقف Pool50/100؛ ۲٫۲ فضای بهبود ترتیب؛ ۲٫۳ جداسازی مؤلفه‌های منبع. شرط پایان: گزارش همهٔ Development و نبود اشتراک منبع میان دو بخش.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [5]:
# NB22C-C005 — Action 2: coverage ceilings, rank losses and source components.
def action02():
    dev = STATE['memory']['dev']; rows = []
    for q in dev['ids']:
        gold = dev['gold'][q]; h = dev['hybrid'][q]
        n50, n100 = len(gold & set(h[:50])), len(gold & set(h))
        ceiling50, ceiling100 = min(10, n50) / len(gold), min(10, n100) / len(gold)
        inc = metrics(dev['incumbent'][q], gold)
        category = ('NO_GOLD_IN_HYBRID100' if not n100 else
                    'GOLD_ONLY_IN_51_TO_100' if not n50 else
                    'ALL_GOLD_IN_INCUMBENT10' if inc['recall_at_10'] == 1 else 'PARTIAL_COVERAGE_OR_ORDERING')
        rows.append({'candidate_id': q, 'category': category, 'source_types': dev['source_types'][q],
            'gold_count': len(gold), 'pool50_gold': n50, 'pool100_gold': n100,
            'pool50_recall_ceiling': ceiling50, 'pool100_recall_ceiling': ceiling100,
            'corpus_top10_recall_ceiling': min(10, len(gold)) / len(gold),
            'incumbent': inc, 'ordering_headroom_in_pool100': ceiling100 - inc['recall_at_10']})
    component_map = components(dev); parts = partition(component_map)
    for row in rows:
        row['source_component'] = component_map[row['candidate_id']]
        row['internal_partition'] = 'validation' if row['candidate_id'] in set(parts['validation']) else 'tuning'
    summary = {'category_counts': dict(collections.Counter(x['category'] for x in rows)),
        'macro_recall_ceiling50': statistics.fmean(x['pool50_recall_ceiling'] for x in rows),
        'macro_recall_ceiling100': statistics.fmean(x['pool100_recall_ceiling'] for x in rows),
        'mean_ordering_headroom100': statistics.fmean(x['ordering_headroom_in_pool100'] for x in rows),
        'mean_pool_expansion_headroom': statistics.fmean(x['pool100_recall_ceiling'] - x['pool50_recall_ceiling'] for x in rows),
        'threshold_reachable_with_current_pool_by_oracle': statistics.fmean(x['pool100_recall_ceiling'] for x in rows) >= CONFIG['thresholds']['recall_at_10'],
        'oracle_is_not_achievable_model_score': True,
        'source_components': len(set(component_map.values())),
        'partition_query_counts': {k: len(v) for k, v in parts.items()},
        'partition_component_counts': {k: len({component_map[q] for q in v}) for k, v in parts.items()},
        'source_disjoint': not ({component_map[q] for q in parts['tuning']} & {component_map[q] for q in parts['validation']}),
        'validation_scope': 'INTERNAL_PREVIOUSLY_EXPOSED_DEVELOPMENT_NOT_INDEPENDENT_FINAL_HOLDOUT'}
    receipt = save('A02.development_diagnosis', rows, rows=True)
    STATE['memory'].update(diagnosis=summary, partitions=parts, component_map=component_map)
    return {'subtasks': {'2.1': 'MEASURED_POOL_CEILINGS', '2.2': 'MEASURED_ORDERING_HEADROOM',
                         '2.3': 'SOURCE_DISJOINT_INTERNAL_PARTITIONS'}, **summary, 'artifact': receipt}
A02 = run_action(2, action02, (1,))


{
  "action": 2,
  "subtasks": {
    "2.1": "MEASURED_POOL_CEILINGS",
    "2.2": "MEASURED_ORDERING_HEADROOM",
    "2.3": "SOURCE_DISJOINT_INTERNAL_PARTITIONS"
  },
  "category_counts": {
    "ALL_GOLD_IN_INCUMBENT10": 1350,
    "PARTIAL_COVERAGE_OR_ORDERING": 1041,
    "NO_GOLD_IN_HYBRID100": 187,
    "GOLD_ONLY_IN_51_TO_100": 86
  },
  "macro_recall_ceiling50": 0.7614690583440583,
  "macro_recall_ceiling100": 0.808861986986987,
  "mean_ordering_headroom100": 0.13581244935411602,
  "mean_pool_expansion_headroom": 0.04739292864292864,
  "threshold_reachable_with_current_pool_by_oracle": false,
  "oracle_is_not_achievable_model_score": true,
  "source_components": 598,
  "partition_query_counts": {
    "tuning": 1992,
    "validation": 672
  },
  "partition_component_counts": {
    "tuning": 485,
    "validation": 113
  },
  "source_disjoint": true,
  "validation_scope": "INTERNAL_PREVIOUSLY_EXPOSED_DEVELOPMENT_NOT_INDEPENDENT_FINAL_HOLDOUT",
  "artifact": {
    "path": "reports/phase2/

<div dir="rtl" align="right">

### NB22C-C006 — اقدام ۳ — فریز آزمایش

۳٫۱ فرضیه و ۱۸ گزینه؛ ۳٫۲ قاعدهٔ انتخاب و بررسی؛ ۳٫۳ اتصال ورودی و کد. شرط پایان: ثبت طرح قبل از امتیازدهی گزینه‌ها.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [6]:
# NB22C-C006 — Action 3: freeze bounded hypotheses before candidate evaluation.
def action03():
    variants = []
    for k in CONFIG['constants']:
        for weight in CONFIG['weights']:
            variants.append({'name': f'rrf100_k{k}_w{weight:.2f}', 'family': 'rrf100', 'k': k, 'weight': weight})
            for limit in CONFIG['newcomer_limits']:
                variants.append({'name': f'quota100_k{k}_w{weight:.2f}_n{limit}',
                                 'family': 'quota100', 'k': k, 'weight': weight, 'newcomer_limit': limit})
    plan = {'schema_version': CONFIG['version'], 'producer_code_sha256': NOTEBOOK_CODE_HASH,
        'source_inputs': STATE['inputs'], 'configuration': CONFIG, 'variants': variants,
        'hypothesis': 'Pure depth100 reranking did not improve recall or nDCG. Test cached Hybrid100/local100 reciprocal rank fusion, and cap previously unscored-pool entrants in top10 to reduce damage from pool expansion.',
        'baseline': 'NB22B_SELECTED_cached_rrf_k10_w0.75',
        'formula': 'w/(k+local100_rank)+(1-w)/(k+hybrid100_rank)',
        'quota_definition': 'Scan fused order; admit at most N chunks outside original Hybrid50 into top10; fill remaining slots with original-pool chunks. Preserve fused order in top10 and tail.',
        'tie_break': 'chunk_id_ascending', 'metric_rr_scope': 'FIRST50_OF_EACH_RESULT',
        'partitions': STATE['memory']['partitions'],
        'selection_rule': 'Tuning only: highest nDCG@10 among strict nDCG improvements with Recall@10 and Hit@10 no lower than NB22B selected; tie by name. No candidate if none eligible.',
        'validation_rule': 'One frozen tuning winner only; require strict nDCG gain and nondecreasing Recall@10/Hit@10 versus incumbent; otherwise retain incumbent. Never select a runner-up using validation.',
        'independence_warning': 'Both partitions come from previously exposed Development. This is an internal check, not an untouched final holdout.',
        'new_api_calls': 0, 'model_pairs': 0, 'historical_test_evaluation': False,
        'absolute_thresholds_unchanged': CONFIG['thresholds'],
        'health_deferral_preserved': True, 'release_acceptance_granted': False}
    receipt = save('A03.frozen_plan', plan)
    STATE['memory'].update(plan=plan, plan_receipt=receipt)
    return {'subtasks': {'3.1': 'FROZEN_HYPOTHESES', '3.2': 'FROZEN_SELECTION_AND_VALIDATION',
                         '3.3': 'BOUND_INPUTS_AND_CODE'}, 'variant_count': len(variants), 'plan': receipt}
A03 = run_action(3, action03, (1, 2))


{
  "action": 3,
  "subtasks": {
    "3.1": "FROZEN_HYPOTHESES",
    "3.2": "FROZEN_SELECTION_AND_VALIDATION",
    "3.3": "BOUND_INPUTS_AND_CODE"
  },
  "variant_count": 18,
  "plan": {
    "path": "reports/phase2/quality_improvement/NB22C.v1/A03.frozen_plan.ab0275b9c15b8447c6508c8a1390a43a64d5f7b1e410ee817e7c7475cd306a89.json",
    "file_bytes_sha256": "ab0275b9c15b8447c6508c8a1390a43a64d5f7b1e410ee817e7c7475cd306a89",
    "bytes": 131439,
    "reload_validation": "PASS"
  },
  "status": "COMPLETE_IN_SCOPE",
  "new_api_calls": 0
}


<div dir="rtl" align="right">

### NB22C-C007 — اقدام ۴ — اجرای آزمایش

۴٫۱ اجرای تمام گزینه‌های ثابت؛ ۴٫۲ مقایسه روی tuning؛ ۴٫۳ فریز یک برنده یا نتیجهٔ منفی. شرط پایان: ثبت انتخاب قبل از مقایسهٔ validation.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [7]:
# NB22C-C007 — Action 4: execute cached-rank experiments and freeze one winner.
def action04():
    dev = STATE['memory']['dev']; plan = STATE['memory']['plan']
    tuning = STATE['memory']['partitions']['tuning']
    reference = mean_metric(metrics(dev['incumbent'][q], dev['gold'][q]) for q in tuning)
    comparisons = []; all_metrics = {}; orders = {}
    started = time.monotonic()
    for spec in plan['variants']:
        name = spec['name']; values = {}; ranked = {}
        for q in dev['ids']:
            order = derived_order(dev['hybrid'][q], dev['local'][q], dev['incumbent'][q], spec)
            ranked[q] = order; values[q] = metrics(order, dev['gold'][q])
        observed = mean_metric(values[q] for q in tuning)
        comparisons.append({'variant': name, 'metrics': observed,
            'delta_from_nb22b_selected': {k: observed[k] - reference[k] for k in reference},
            'eligible': eligible(observed, reference), 'query_count': len(tuning)})
        all_metrics[name] = values; orders[name] = ranked
        print(json.dumps({'stage': 'CACHED_EXPERIMENT', 'variant': name,
            'tuning_recall_at_10': observed['recall_at_10'], 'tuning_ndcg_at_10': observed['ndcg_at_10'],
            'elapsed_seconds': round(time.monotonic() - started, 2), 'new_api_calls': 0}), flush=True)
    allowed = [x for x in comparisons if x['eligible']]
    winner = sorted(allowed, key=lambda x: (-x['metrics']['ndcg_at_10'], x['variant']))[0]['variant'] if allowed else None
    freeze = {'plan': STATE['memory']['plan_receipt'], 'selected_variant': winner,
        'selection_scope': 'TUNING_PARTITION_ONLY', 'reference_metrics': reference,
        'tuning_comparisons': comparisons, 'validation_used_for_selection': False}
    receipt = save('A04.tuning_selection_freeze', freeze)
    # Winner and all tuning comparisons are on disk before validation comparison.
    STATE['memory'].update(variant_metrics=all_metrics, orders=orders, winner=winner, selection_receipt=receipt)
    return {'subtasks': {'4.1': 'EXECUTED_ALL_FIXED_VARIANTS', '4.2': 'COMPARED_TUNING_ONLY',
                         '4.3': 'FROZEN_ONE_WINNER_OR_NEGATIVE_RESULT'},
            'selected_variant': winner, 'reference_tuning': reference, 'tuning_comparisons': comparisons,
            'freeze': receipt, 'new_local_model_pairs': 0}
A04 = run_action(4, action04, (1, 2, 3))


{"stage": "CACHED_EXPERIMENT", "variant": "rrf100_k10_w0.50", "tuning_recall_at_10": 0.6714435121438134, "tuning_ndcg_at_10": 0.5991647870462554, "elapsed_seconds": 0.34, "new_api_calls": 0}
{"stage": "CACHED_EXPERIMENT", "variant": "quota100_k10_w0.50_n1", "tuning_recall_at_10": 0.6728240342321667, "tuning_ndcg_at_10": 0.5996415409116923, "elapsed_seconds": 0.74, "new_api_calls": 0}
{"stage": "CACHED_EXPERIMENT", "variant": "quota100_k10_w0.50_n2", "tuning_recall_at_10": 0.6719455201759419, "tuning_ndcg_at_10": 0.5993427381374041, "elapsed_seconds": 1.18, "new_api_calls": 0}
{"stage": "CACHED_EXPERIMENT", "variant": "rrf100_k10_w0.75", "tuning_recall_at_10": 0.6734995537706381, "tuning_ndcg_at_10": 0.60961010981169, "elapsed_seconds": 1.55, "new_api_calls": 0}
{"stage": "CACHED_EXPERIMENT", "variant": "quota100_k10_w0.75_n1", "tuning_recall_at_10": 0.6746673200739466, "tuning_ndcg_at_10": 0.6103885559608998, "elapsed_seconds": 1.96, "new_api_calls": 0}
{"stage": "CACHED_EXPERIMENT", "

<div dir="rtl" align="right">

### NB22C-C008 — اقدام ۵ — بررسی داخلی

۵٫۱ فقط یک برندهٔ فریز‌شده؛ ۵٫۲ زیرگروه، عدم‌قطعیت و موارد افت؛ ۵٫۳ حفظ منتخب قبلی در صورت شکست. شرط پایان: خروجی رتبه‌های حفظ‌شده و نتیجهٔ صریح؛ انتخاب نفر بعد با validation ممنوع.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [8]:
# NB22C-C008 — Action 5: assess one frozen candidate on internal validation.
def paired_component_intervals(ids, candidate, reference, component_map):
    import random
    names = ['recall_at_10', 'ndcg_at_10', 'hit_at_10']
    groups = collections.defaultdict(list)
    for q in ids:
        groups[component_map[q]].append(q)
    ordered = [groups[c] for c in sorted(groups)]
    sizes = [len(g) for g in ordered]
    sums = [[math.fsum(candidate[q][k] - reference[q][k] for q in g) for k in names] for g in ordered]
    rng = random.Random(CONFIG['seed']); samples = {k: [] for k in names}; repetitions = 2000
    for _ in range(repetitions):
        chosen = [rng.randrange(len(ordered)) for _ in ordered]
        denominator = sum(sizes[i] for i in chosen)
        for j, name in enumerate(names):
            samples[name].append(math.fsum(sums[i][j] for i in chosen) / denominator)
    def quantile(values, p):
        values = sorted(values); position = (len(values) - 1) * p
        lo = int(position); hi = min(lo + 1, len(values) - 1)
        return values[lo] + (values[hi] - values[lo]) * (position - lo)
    return {'method': 'PAIRED_SOURCE_COMPONENT_BOOTSTRAP', 'repetitions': repetitions,
        'component_count': len(ordered), 'confidence': 0.95,
        'scope': 'INTERNAL_EXPLORATORY_PREVIOUSLY_EXPOSED_DEVELOPMENT',
        'familywise_significance_claimed': False,
        'intervals': {k: {'lower': quantile(v, .025), 'upper': quantile(v, .975)} for k, v in samples.items()}}

def action05():
    dev = STATE['memory']['dev']; winner = STATE['memory']['winner']
    validation = STATE['memory']['partitions']['validation']
    base_values = {q: metrics(dev['incumbent'][q], dev['gold'][q]) for q in dev['ids']}
    before = mean_metric(base_values[q] for q in validation)
    candidate_values = STATE['memory']['variant_metrics'][winner] if winner else base_values
    after = mean_metric(candidate_values[q] for q in validation)
    passed = winner is not None and eligible(after, before)
    retained = winner if passed else 'NB22B_SELECTED_cached_rrf_k10_w0.75'
    retained_values = candidate_values if passed else base_values
    slices = {}
    labels = sorted({','.join(dev['source_types'][q]) for q in validation})
    for label in labels:
        qs = [q for q in validation if ','.join(dev['source_types'][q]) == label]
        b = mean_metric(base_values[q] for q in qs); a = mean_metric(candidate_values[q] for q in qs)
        slices[label] = {'queries': len(qs), 'before': b, 'candidate': a,
                         'delta': {k: a[k] - b[k] for k in b}}
    errors = []
    selected_orders = STATE['memory']['orders'][winner] if winner else dev['incumbent']
    for q in validation:
        delta = candidate_values[q]['recall_at_10'] - base_values[q]['recall_at_10']
        if abs(delta) > 1e-12:
            errors.append({'candidate_id': q, 'recall_at_10_delta': delta,
                'source_types': dev['source_types'][q], 'gold_count': len(dev['gold'][q]),
                'incumbent_top10': dev['incumbent'][q][:10], 'candidate_top10': selected_orders[q][:10],
                'relevant_chunk_ids': sorted(dev['gold'][q])})
    error_receipt = save('A05.validation_changed_queries', sorted(errors, key=lambda r: (r['recall_at_10_delta'], r['candidate_id'])), rows=True)
    output_rows = [{'candidate_id': q, 'split': 'development', 'variant': retained,
                   'ranked_chunk_ids': (selected_orders if passed else dev['incumbent'])[q],
                   'metrics': retained_values[q]} for q in dev['ids']]
    rank_receipt = save('A05.retained_development_rankings', output_rows, rows=True)
    summary = {'frozen_tuning_candidate': winner, 'retained_variant': retained,
        'candidate_passes_internal_guard': passed, 'validation_queries': len(validation),
        'validation_before': before, 'validation_candidate': after,
        'validation_delta': {k: after[k] - before[k] for k in before},
        'internal_uncertainty': paired_component_intervals(validation, candidate_values, base_values, STATE['memory']['component_map']) if winner else None,
        'validation_source_slices': slices,
        'full_development_before': mean_metric(base_values.values()),
        'full_development_retained': mean_metric(retained_values.values()),
        'validation_changes': {'improved_recall_queries': sum(x['recall_at_10_delta'] > 0 for x in errors),
                               'worsened_recall_queries': sum(x['recall_at_10_delta'] < 0 for x in errors),
                               'unchanged_recall_queries': len(validation) - len(errors)},
        'changed_queries': error_receipt, 'retained_rankings': rank_receipt,
        'independent_final_acceptance': False, 'historical_test_evaluation': False,
        'internal_guard_is_point_rule_not_statistical_noninferiority_proof': True}
    receipt = save('A05.internal_validation', summary)
    STATE['memory'].update(validation=summary, validation_receipt=receipt)
    return {'subtasks': {'5.1': 'EVALUATED_ONE_FROZEN_WINNER', '5.2': 'REPORTED_SLICES_UNCERTAINTY_AND_ERRORS',
                         '5.3': 'PRESERVED_INCUMBENT_ON_GUARD_FAILURE'}, **summary, 'report': receipt}
A05 = run_action(5, action05, (1, 2, 3, 4))


{
  "action": 5,
  "subtasks": {
    "5.1": "EVALUATED_ONE_FROZEN_WINNER",
    "5.2": "REPORTED_SLICES_UNCERTAINTY_AND_ERRORS",
    "5.3": "PRESERVED_INCUMBENT_ON_GUARD_FAILURE"
  },
  "frozen_tuning_candidate": "quota100_k10_w0.75_n1",
  "retained_variant": "quota100_k10_w0.75_n1",
  "candidate_passes_internal_guard": true,
  "validation_queries": 672,
  "validation_before": {
    "hit_at_10": 0.8229166666666666,
    "hit_at_5": 0.78125,
    "ndcg_at_10": 0.6030530652457177,
    "ndcg_at_5": 0.5800917414128289,
    "precision_at_10": 0.10476190476190476,
    "precision_at_5": 0.1875,
    "recall_at_10": 0.6839037698412698,
    "recall_at_5": 0.6238839285714286,
    "reciprocal_rank": 0.6663446830456058
  },
  "validation_candidate": {
    "hit_at_10": 0.8303571428571429,
    "hit_at_5": 0.7872023809523809,
    "ndcg_at_10": 0.6037513390657998,
    "ndcg_at_5": 0.5801875662766163,
    "precision_at_10": 0.1055059523809524,
    "precision_at_5": 0.18839285714285717,
    "recall_at_10": 

<div dir="rtl" align="right">

### NB22C-C009 — اقدام ۶ — گزارش نهایی

۶٫۱ بازخوانی شواهد خروجی؛ ۶٫۲ عدد قبل/بعد و حد مطلق؛ ۶٫۳ موانع و گام بعدی. شرط پایان: گزارش تصمیم‌پذیر؛ پذیرش مستقل محصول همچنان نیازمند شاهد جداست.

بستهٔ کامل همین مرحله؛ API و مدل تازه صفر. پیش‌نیاز نامعتبر مرحلهٔ وابسته را مسدود می‌کند.

</div>

In [9]:
# NB22C-C009 — Action 6: concrete final decision, unresolved work and evidence.
def action06():
    require(all(STATE['actions'].get(i, {}).get('status') == 'COMPLETE_IN_SCOPE' for i in range(1, 6)), 'EARLIER_ACTIONS_NOT_COMPLETE')
    summary = STATE['memory']['validation']; diagnosis = STATE['memory']['diagnosis']
    retained = summary['full_development_retained']
    threshold_checks = {k: {'value': retained[k], 'threshold': v,
        'status': 'PASS' if retained[k] >= v else 'FAIL', 'gap_to_threshold': max(0.0, v - retained[k])}
        for k, v in CONFIG['thresholds'].items()}
    pool_limited = diagnosis['macro_recall_ceiling100'] < CONFIG['thresholds']['recall_at_10']
    next_step = ('NB16_UPSTREAM_CANDIDATE_EXPANSION_REQUIRED_FOR_RECALL_TARGET_IN_THIS_POOL' if pool_limited else
                 'NB17_ORDERING_OR_GOLD_QUALIFICATION_INVESTIGATION_BEFORE_NEW_MODEL_RUN')
    final = {'schema_version': 'NB22C_FINAL_V1',
        'status': 'INTERNAL_RETRIEVAL_EXPERIMENT_COMPLETE_ACCEPTANCE_NOT_ESTABLISHED',
        'retained_variant': summary['retained_variant'],
        'new_variant_passes_internal_guard': summary['candidate_passes_internal_guard'],
        'before': summary['full_development_before'], 'after': retained,
        'change': {k: retained[k] - summary['full_development_before'][k] for k in retained},
        'development_absolute_threshold_checks': threshold_checks,
        'diagnosis': diagnosis, 'next_step': next_step,
        'next_step_conditions': {
            'if_pool_ceiling_below_target': 'Diagnose dense/sparse retrieval inputs and candidate coverage. Reranking alone cannot exceed the measured pool ceiling.',
            'if_pool_ceiling_sufficient_but_ranking_fails': 'Inspect changed-query catalogue and pinned reranker truncation/tokenization/scoring; define a small representative local-model pilot before expensive inference.',
            'gold_changes': 'Only independently proven label/mapping errors, versioned; never adjust labels to improve metrics.',
            'independent_evaluation': 'Requires a qualified new or genuinely untouched holdout before final acceptance.',
            'product_quality': 'Requires binding chosen retrieval to actual context/answer generation and semantic/safety/regression evidence.'},
        'preserved_open_work': ['NB22B semantic reviews: 261 held responses and one HTTP504 reported in current chat; not reopened here',
            'Product generation and qualified groundedness/completeness/citations/safety remain separate',
            'NB21 action48 and deferred health/consolidation remain deferred',
            'Historical FAIL/BLOCKED and release nonacceptance are preserved'],
        'historical_test_evaluated': False, 'new_api_calls': 0, 'new_model_pairs': 0,
        'historical_files_modified': False, 'release_acceptance_granted': False,
        'producer_code_sha256': NOTEBOOK_CODE_HASH,
        'environment': {'python': platform.python_version(), 'platform': platform.system()},
        'actions_1_to_5': {str(k): v for k, v in STATE['actions'].items()}, 'source_inputs': STATE['inputs'],
        'artifacts': list(STATE['artifacts'])}
    for r in final['artifacts']:
        require(digest(path_for(r['path']).read_bytes()) == r['file_bytes_sha256'], 'FINAL_ARTIFACT_HASH_CONFLICT')
    receipt = save('A06.final_report', final)
    return {'subtasks': {'6.1': 'VERIFIED_OUTPUT_EVIDENCE', '6.2': 'REPORTED_REAL_BEFORE_AFTER',
                         '6.3': 'PRESERVED_OPEN_GATES_AND_NEXT_STEP'},
        'retained_variant': final['retained_variant'], 'before': final['before'], 'after': final['after'],
        'change': final['change'], 'threshold_checks': threshold_checks,
        'new_variant_passes_internal_guard': summary['candidate_passes_internal_guard'],
        'next_step': next_step, 'release_acceptance_granted': False, 'report': receipt}
A06 = run_action(6, action06)
print('\n========== NB22C — نتیجهٔ نهایی ==========')
if A06['status'] != 'COMPLETE_IN_SCOPE':
    print(json.dumps({'status': 'INCOMPLETE_NO_ACCEPTANCE', 'actions': STATE['actions'],
        'instruction_fa': 'فقط فایل یا قرارداد مشخص‌شده در علت توقف را بررسی کن؛ NB22B و API را دوباره اجرا نکن.',
        'release_acceptance_granted': False, 'new_api_calls': 0}, ensure_ascii=False, indent=2))
else:
    print(json.dumps({'status': 'EXPERIMENT_COMPLETE_NOT_RELEASE_ACCEPTED',
        'retained_variant': A06['retained_variant'], 'change': A06['change'],
        'threshold_checks': A06['threshold_checks'], 'next_step': A06['next_step'],
        'report': A06['report'], 'new_api_calls': 0}, ensure_ascii=False, indent=2))


{
  "action": 6,
  "subtasks": {
    "6.1": "VERIFIED_OUTPUT_EVIDENCE",
    "6.2": "REPORTED_REAL_BEFORE_AFTER",
    "6.3": "PRESERVED_OPEN_GATES_AND_NEXT_STEP"
  },
  "retained_variant": "quota100_k10_w0.75_n1",
  "before": {
    "hit_at_10": 0.8506006006006006,
    "hit_at_5": 0.8025525525525525,
    "ndcg_at_10": 0.6069571673947197,
    "ndcg_at_5": 0.5844806844946681,
    "precision_at_10": 0.11636636636636637,
    "precision_at_5": 0.2063813813813814,
    "recall_at_10": 0.6730495376328709,
    "recall_at_5": 0.6117641450974785,
    "reciprocal_rank": 0.697180277623896
  },
  "after": {
    "hit_at_10": 0.8584834834834835,
    "hit_at_5": 0.8066816816816816,
    "ndcg_at_10": 0.6087143030504241,
    "ndcg_at_5": 0.5846432740221266,
    "precision_at_10": 0.1170795795795796,
    "precision_at_5": 0.2063813813813814,
    "recall_at_10": 0.6779356737690071,
    "recall_at_5": 0.6121113673197007,
    "reciprocal_rank": 0.699319256255785
  },
  "change": {
    "hit_at_10": 0.0078828828

<div dir="rtl" align="right">

## حدود اعتبارسنجی تحویل

۲۸ کنترل نرم‌افزاری روی دادهٔ مصنوعی پاس شده: Schema نوت‌بوک، محاسبهٔ مستقل سنجه، سهمیهٔ ورودی تازه، تفکیک منبع، اجرای ترتیبی ۹ سل کد در فضای تازه، تکرار امن، ورودی ناقص/خراب و حالت نتیجهٔ منفی یا افت validation. در این آزمون‌ها شبکه مسدود بود.

این گزارش اجرای واقعی داده‌های پروژه یا اجرای Kernel خود دستگاه کاربر نیست. گزارش C056 و فایل ۱۴۰ مگابایتی A05 در محیط تهیهٔ این نسخه موجود نبودند؛ اجرای واقعی و عدد بهبود فقط بعد از اجرای همین فایل در مخزن با ورودی‌های پین‌شده معلوم می‌شود.

</div>